In [1]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain.chat_models import init_chat_model
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
import os
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader


In [2]:
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

/tmp/ipykernel_1115/954509828.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.1"
MODEL_PATH = "/content/drive/MyDrive/mistral_7b_4bit"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

if not os.path.exists(MODEL_PATH) or len(os.listdir(MODEL_PATH)) == 0:

    print("Model not found. Downloading for first time...")

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True,
        use_safetensors=True
    )

    os.makedirs(MODEL_PATH, exist_ok=True)

    tokenizer.save_pretrained(MODEL_PATH)
    model.save_pretrained(MODEL_PATH)

    print("Model downloaded and saved to Drive!")

else:
    print("Model found in Drive. Loading...")

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_PATH,
        trust_remote_code=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        quantization_config=quantization_config,
        device_map="auto",
        trust_remote_code=True
    )

    print("Model loaded successfully from Drive!")

Model found in Drive. Loading...


/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:246: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded successfully from Drive!


In [4]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_1115/96220787.py:12: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [5]:
loader = TextLoader("/content/Chennai real estate.txt")
documents = loader.load()

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

docs = text_splitter.split_documents(documents)

In [7]:
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [46]:
def self_rag(question: str):
    """
    Self-RAG function with:
    - Property Type + Locality exact matching
    - Critic validates extraction
    - Final Answer is chatbot-style
    - Only 3 lines output
    """
    # STEP 0 — Retrieve documents
    context_docs = retriever.invoke(question)
    if not context_docs:
        print("Initial Answer: Average Price not found in data.")
        print("Critique's Answer: NO. No relevant documents found.")
        print("🔍 Final Answer:")
        print("Sorry, I could not find the average price for your query.")
        return

    # Combine retrieved text
    context = "\n".join(doc.page_content for doc in context_docs)

    # STEP 1 — Identify Property Type and Locality from the question
    property_types = ["Multistorey Apartment", "Builder Floor Apartment",
                      "Residential Plots", "Residential House"]
    property_type = next((pt for pt in property_types if pt.lower() in question.lower()), None)

    # Try to find a locality in the retrieved context
    # We assume each line is like: Locality: <name>
    localities = re.findall(r"Locality\s*:\s*(.*)", context)
    locality = next((loc for loc in localities if loc.lower() in question.lower()), None)

    # STEP 2 — Extract Initial Answer
    initial_answer = "Average Price not found in data."
    final_value = None

    if property_type and locality:
        # Look for line containing property_type and locality
        pattern = rf"Property Type\s*:\s*{re.escape(property_type)}.*?Locality\s*:\s*{re.escape(locality)}.*?Average Price\s*:\s*(\d+\s*Rs\/sq\.(?:ft|Yrd))"
        match = re.search(pattern, context, flags=re.DOTALL | re.IGNORECASE)
        if match:
            initial_answer = match.group(1)
            final_value = initial_answer

    # STEP 3 — Critic
    if final_value:
        critique = "YES. The answer matches the correct property and locality."
    else:
        critique = "NO. Initial extraction failed or incorrect."
        # Attempt to correct by scanning all price lines for the locality
        corrected_match = re.search(rf"Locality\s*:\s*{re.escape(locality)}.*?Average Price\s*:\s*(\d+\s*Rs\/sq\.(?:ft|Yrd))", context, flags=re.DOTALL | re.IGNORECASE)
        if corrected_match:
            final_value = corrected_match.group(1)
            initial_answer = final_value  # Update initial answer for output

    # STEP 4 — Final Answer (chatbot-style)
    if final_value:
        final_answer = f"The average price for {locality} {property_type} is {final_value}."
    else:
        final_answer = "Sorry, I could not find the average price for your query."

    # Print only 3 lines
    print(f"Initial Answer: {initial_answer}")
    print(f"Critique's Answer: {critique}")
    print("🔍 Final Answer:")
    print(final_answer)

    return final_answer

In [50]:
questions = [
    "Average price for Tambaram Multistorey Apartment?"
]

for q in questions:
    self_rag(q)

Initial Answer: 6132 Rs/sq.ft
Critique's Answer: YES. The answer matches the correct property and locality.
🔍 Final Answer:
The average price for Tambaram Multistorey Apartment is 6132 Rs/sq.ft.
